# S2 orbit fit -- Quadratic Gravity

This notebook runs the full pipeline (optimize -> MCMC -> diagnostics) for one metric.
All the physics and priors live in one file, `snope/metrics/quadratic_gravity.py` -- edit that file, not this notebook, to change the model.

In [ ]:
from snope.metrics.quadratic_gravity import main, metric, param_priors
from snope.priors import PriorMode
from snope.orbit_model import S2OrbitModel
from snope import plotting


## 1. Sanity check: does this metric give a bound orbit?

Before spending time on a full MCMC run, check that the effective potential
`V_eff(r)` actually has two turning points (periapsis and apoapsis) at the
trial parameter values -- if it doesn't, the orbit isn't bound and the full
integrator will fail or return garbage. This takes a fraction of a second,
no orbit integration required.

In [ ]:
model = S2OrbitModel(metric, "../data/tab_gillessen_pos.csv", "../data/tab_gillessen_vr.csv", verbose=False)

preview = model.preview_effective_potential(
    M_bh=4.3e6, distance=8.33, a=125.5, e=0.884,
    k=-1.0,
)
plotting.effective_potential_plot(preview, "Quadratic Gravity")

Try a few different values of the new parameter here (and of `a`/`e`) if you're
not sure what range makes sense physically -- it's much cheaper to explore that
here than inside the MCMC prior bounds.

## 2. Run the fit

Defaults use mixed priors: flat on the orbital elements and the new-physics
parameter (`k`), Gaussian on the offsets and `t_peri`.

In [ ]:
sampler, flat_samples, best_fit = main(
    data_path_pos="../data/tab_gillessen_pos.csv",
    data_path_rv="../data/tab_gillessen_vr.csv",
)
best_fit

## Other prior modes and settings

```python
# every parameter flat
main(prior_mode=PriorMode.FLAT)

# every parameter Gaussian
main(prior_mode=PriorMode.GAUSSIAN)

# override any pipeline setting (n_steps, n_walkers, n_points, ...)
main(prior_mode=PriorMode.MIXED, n_walkers=64, n_steps=50000, burn_in=5000)

# quick local run to sanity-check everything works, before scaling up
main(n_walkers=8, n_steps=200, burn_in=20, n_points=200)
```